# Network Intrusion Detection Using Random Forest and Deep Learning
**NSL-KDD | Random Forest + Deep Neural Network + Hybrid (soft-voting) | B.Tech CSE Mini Project**

Pipeline: Load data -> Label mapping -> Preprocessing -> Feature selection ->
Train RF and DNN -> Hybrid ensemble -> Evaluate (Accuracy / Precision / Recall / F1)

Run the cells from top to bottom.

## 1. Setup

In [ ]:
import os, sys, time, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
sns.set_theme(style="whitegrid")

DATA_DIR = "data"          # KDDTrain+.txt and KDDTest+.txt go here
OUT_DIR = "outputs"        # plots, tables and saved models are written here
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print("TensorFlow:", tf.__version__)

## 2. Load the NSL-KDD dataset
Needs `KDDTrain+.txt` (125,973 rows) and `KDDTest+.txt` (22,544 rows).
If they are not in `data/`, the cell tries to download them; if that fails,
on Google Colab it opens a file-upload dialog (get the files from
https://www.kaggle.com/datasets/hassan06/nslkdd).

In [ ]:
TRAIN_PATH = os.path.join(DATA_DIR, "KDDTrain+.txt")
TEST_PATH = os.path.join(DATA_DIR, "KDDTest+.txt")

MIRRORS = [  # tried in order; ignored if unreachable
    "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/{}",
]

def ensure_dataset():
    import urllib.request, urllib.parse
    for fname, path in [("KDDTrain+.txt", TRAIN_PATH), ("KDDTest+.txt", TEST_PATH)]:
        if os.path.exists(path):
            continue
        for base in MIRRORS:
            url = base.format(urllib.parse.quote(fname))
            try:
                print("Downloading", url)
                urllib.request.urlretrieve(url, path)
                break
            except Exception as e:
                print("  failed:", e)
    if not (os.path.exists(TRAIN_PATH) and os.path.exists(TEST_PATH)):
        try:  # Google Colab manual upload
            from google.colab import files
            print("Please upload KDDTrain+.txt and KDDTest+.txt")
            for name, content in files.upload().items():
                with open(os.path.join(DATA_DIR, name), "wb") as f:
                    f.write(content)
        except ImportError:
            pass
    if not (os.path.exists(TRAIN_PATH) and os.path.exists(TEST_PATH)):
        sys.exit(f"Dataset missing. Put KDDTrain+.txt and KDDTest+.txt in '{DATA_DIR}/'.")

ensure_dataset()

COLUMNS = ["duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
           "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
           "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
           "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
           "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
           "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
           "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
           "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
           "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
           "label", "difficulty"]

train_df = pd.read_csv(TRAIN_PATH, names=COLUMNS)
test_df = pd.read_csv(TEST_PATH, names=COLUMNS)
train_df.drop(columns="difficulty", inplace=True)
test_df.drop(columns="difficulty", inplace=True)
print("Train:", train_df.shape, "| Test:", test_df.shape)
train_df.head()

## 3. Map attack names to the 5 traffic classes (Normal, DoS, Probe, R2L, U2R)

In [ ]:
ATTACK_MAP = {"normal": "Normal"}
for a in ["back", "land", "neptune", "pod", "smurf", "teardrop", "apache2", "udpstorm",
          "processtable", "mailbomb", "worm"]:
    ATTACK_MAP[a] = "DoS"
for a in ["satan", "ipsweep", "nmap", "portsweep", "mscan", "saint"]:
    ATTACK_MAP[a] = "Probe"
for a in ["guess_passwd", "ftp_write", "imap", "phf", "multihop", "warezmaster", "warezclient",
          "spy", "xlock", "xsnoop", "snmpguess", "snmpgetattack", "httptunnel", "sendmail", "named"]:
    ATTACK_MAP[a] = "R2L"
for a in ["buffer_overflow", "loadmodule", "rootkit", "perl", "sqlattack", "xterm", "ps"]:
    ATTACK_MAP[a] = "U2R"

for df in (train_df, test_df):
    df["attack_class"] = df["label"].map(ATTACK_MAP)
    df["is_attack"] = (df["attack_class"] != "Normal").astype(int)   # 0 = Normal, 1 = Malicious
    if df["attack_class"].isna().any():
        print("Unmapped labels dropped:", df.loc[df["attack_class"].isna(), "label"].unique())
        df.dropna(subset=["attack_class"], inplace=True)

CLASS_NAMES = ["Normal", "DoS", "Probe", "R2L", "U2R"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}

dist = train_df["attack_class"].value_counts()
print((dist / len(train_df) * 100).round(2).to_string())

plt.figure(figsize=(6, 4))
sns.barplot(x=dist.index, y=dist.values, palette="viridis")
plt.title("Class distribution (training set)"); plt.ylabel("Records")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/class_distribution.png", dpi=150); plt.show()

## 4. Preprocessing
* One-hot encoding of the 3 categorical features (`protocol_type`, `service`, `flag`)
* Min-Max normalisation (fit on training data only)
* Hold out 20% of the training data for validation / early stopping

In [ ]:
CAT_COLS = ["protocol_type", "service", "flag"]
FEATURES = [c for c in COLUMNS if c not in ("label", "difficulty")]

# Encode train+test together so both have identical columns
full = pd.concat([train_df[FEATURES], test_df[FEATURES]], keys=["train", "test"])
full = pd.get_dummies(full, columns=CAT_COLS, dtype=np.float32)
X_train_all = full.loc["train"].reset_index(drop=True)
X_test_all = full.loc["test"].reset_index(drop=True)
print("Features after one-hot encoding:", X_train_all.shape[1])

scaler = MinMaxScaler().fit(X_train_all)
X_train_all = pd.DataFrame(scaler.transform(X_train_all), columns=X_train_all.columns).astype(np.float32)
X_test_all = pd.DataFrame(scaler.transform(X_test_all), columns=X_test_all.columns).astype(np.float32)
X_test_all = X_test_all.clip(0, 1)   # unseen test values can fall slightly outside the train range

## 5. Feature selection (Random Forest importance)

In [ ]:
N_SELECT = 40   # number of features to keep

selector = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED)
selector.fit(X_train_all, train_df["attack_class"].map(CLASS_TO_ID))
importances = pd.Series(selector.feature_importances_, index=X_train_all.columns).sort_values(ascending=False)
selected_features = importances.head(N_SELECT).index.tolist()
print(f"Kept top {N_SELECT} of {len(importances)} features "
      f"({importances.head(N_SELECT).sum() * 100:.1f}% of total importance)")

plt.figure(figsize=(8, 5))
importances.head(15)[::-1].plot(kind="barh", color="#3b6ea5")
plt.title("Top 15 features (Random Forest importance)"); plt.xlabel("Importance")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/feature_importance.png", dpi=150); plt.show()

X_train_sel = X_train_all[selected_features]
X_test_sel = X_test_all[selected_features]

## 6. Model definitions and evaluation helpers

In [ ]:
def build_dnn(input_dim, n_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"), layers.BatchNormalization(), layers.Dropout(0.3),
        layers.Dense(64, activation="relu"), layers.BatchNormalization(), layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def score(y_true, y_pred, name, split):
    return {"Model": name, "Split": split,
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "F1-Score": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "Macro-F1": f1_score(y_true, y_pred, average="macro", zero_division=0)}


def plot_cm(y_true, y_pred, labels, title, path):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels,
                cbar_kws={"label": "Row-normalised"})
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(title)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()


def run_experiment(mode):
    """mode = 'multiclass' (5 classes) or 'binary' (Normal vs Malicious)."""
    print("\n" + "=" * 70 + f"\n  EXPERIMENT: {mode.upper()}\n" + "=" * 70)
    if mode == "binary":
        labels = ["Normal", "Malicious"]
        y_all = train_df["is_attack"].values
        y_test = test_df["is_attack"].values
    else:
        labels = CLASS_NAMES
        y_all = train_df["attack_class"].map(CLASS_TO_ID).values
        y_test = test_df["attack_class"].map(CLASS_TO_ID).values
    n_classes = len(labels)

    # 80/20 stratified split of the training set (validation used for early stopping)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_sel.values, y_all, test_size=0.2, stratify=y_all, random_state=SEED)
    X_te = X_test_sel.values

    # ---- Random Forest ----
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
    rf.fit(X_tr, y_tr)
    print(f"Random Forest trained in {time.time() - t0:.1f}s")
    rf_val, rf_te = rf.predict_proba(X_val), rf.predict_proba(X_te)

    # ---- Deep Neural Network ----
    cw = None
    if mode == "multiclass":   # soften class weights so rare classes (R2L/U2R) get more attention
        w = compute_class_weight("balanced", classes=np.arange(n_classes), y=y_tr)
        cw = {i: float(v ** 0.5) for i, v in enumerate(w)}
    dnn = build_dnn(X_tr.shape[1], n_classes)
    t0 = time.time()
    hist = dnn.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=40, batch_size=256,
                   class_weight=cw, verbose=2,
                   callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                      restore_best_weights=True)])
    print(f"DNN trained in {time.time() - t0:.1f}s")
    dnn_val, dnn_te = dnn.predict(X_val, verbose=0), dnn.predict(X_te, verbose=0)

    # ---- Hybrid: average the class probabilities of RF and DNN ----
    hy_val, hy_te = (rf_val + dnn_val) / 2, (rf_te + dnn_te) / 2

    preds = {"Random Forest": (rf_val.argmax(1), rf_te.argmax(1)),
             "Deep Learning (DNN)": (dnn_val.argmax(1), dnn_te.argmax(1)),
             "Hybrid (RF + DNN)": (hy_val.argmax(1), hy_te.argmax(1))}

    rows = []
    for name, (pv, pt) in preds.items():
        rows.append(score(y_val, pv, name, "Validation (20% of train)"))
        rows.append(score(y_test, pt, name, "KDDTest+"))
    results = pd.DataFrame(rows)

    # ---- Reports ----
    for name, (_, pt) in preds.items():
        print(f"\n--- {name} on KDDTest+ ---")
        print(classification_report(y_test, pt, labels=range(n_classes), target_names=labels,
                                    digits=4, zero_division=0))
        plot_cm(y_test, pt, labels, f"{name} - {mode} (KDDTest+)",
                f"{OUT_DIR}/cm_{mode}_{name.split()[0].lower()}.png")

    # ---- Training curves ----
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val")
    ax[0].set_title("DNN loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
    ax[1].plot(hist.history["accuracy"], label="train"); ax[1].plot(hist.history["val_accuracy"], label="val")
    ax[1].set_title("DNN accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/dnn_curves_{mode}.png", dpi=150); plt.show()

    results.to_csv(f"{OUT_DIR}/results_{mode}.csv", index=False)
    return results, rf, dnn, hy_te

## 7. Experiment A - Binary classification (Normal vs Malicious)
This is the main goal stated in the abstract.

In [ ]:
res_bin, rf_bin, dnn_bin, hy_te_bin = run_experiment("binary")

## 8. Experiment B - Five-class classification (Normal / DoS / Probe / R2L / U2R)

In [ ]:
res_multi, rf_multi, dnn_multi, hy_te_multi = run_experiment("multiclass")

## 9. Final comparison table and chart (use these for your review)

In [ ]:
pd.set_option("display.width", 200)
for title, res in [("BINARY (Normal vs Malicious)", res_bin), ("MULTI-CLASS (5 classes)", res_multi)]:
    print(f"\n===== {title} =====")
    print(res.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (title, res) in zip(axes, [("Binary", res_bin), ("Multi-class", res_multi)]):
    t = res[res["Split"] == "KDDTest+"].set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]]
    t.plot(kind="bar", ax=ax, rot=15, colormap="viridis")
    ax.set_title(f"{title} - performance on KDDTest+"); ax.set_ylim(0, 1); ax.set_xlabel("")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/model_comparison.png", dpi=150); plt.show()

## 10. Save the trained models and run a sample prediction

In [ ]:
joblib.dump({"scaler": scaler, "selected_features": selected_features,
             "all_columns": list(X_train_all.columns)}, f"{OUT_DIR}/preprocessing.joblib")
joblib.dump(rf_bin, f"{OUT_DIR}/rf_binary.joblib", compress=3)
dnn_bin.save(f"{OUT_DIR}/dnn_binary.keras")
print("Saved to", os.path.abspath(OUT_DIR))

# Demo: classify 8 random test connections with the hybrid binary model
idx = np.random.RandomState(1).choice(len(test_df), 8, replace=False)
demo = pd.DataFrame({"Actual": np.where(test_df["is_attack"].values[idx] == 1, "Malicious", "Normal"),
                     "Actual type": test_df["label"].values[idx],
                     "Hybrid prediction": np.where(hy_te_bin.argmax(1)[idx] == 1, "Malicious", "Normal"),
                     "Confidence": hy_te_bin.max(1)[idx].round(3)})
print(demo.to_string(index=False))